# 🛰️ SIH26142 — Deep Super-Resolution Fine-Tuning (Google Colab GPU)
### Sponsoring Agency: NTRO | Model: Real-ESRGAN (RRDBNet Backbone)

This notebook runs **Deep Super-Resolution Training** with free NVIDIA T4 GPU acceleration.

### 🔬 Advanced Multi-Loss Suite Included:
1. **$L_1$ Pixel Reconstruction Loss**: Direct radiometric fidelity
2. **VGG-16 Perceptual Loss**: High-level semantic and structural realism
3. **Differentiable Spectral Angle Mapper (SAM) Loss**: Zero spectral distortion across bands
4. **Sobel Gradient Edge Loss**: Razor-sharp linear infrastructure (roads, runways, coastlines)
5. **2D FFT Frequency Loss**: Recovers high-frequency micro-textures
6. **Full $D_4$ Dihedral Augmentations**: 8-way rotation and reflection invariance
7. **FP16 Mixed Precision (AMP)**: 2× faster training on NVIDIA Tensor Cores

---

In [ ]:
# ── Cell 1: Check GPU Acceleration ─────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Model:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: Please change runtime to GPU! (Runtime -> Change runtime type -> T4 GPU)')

In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────
!pip install -q basicsr facexlib gfpgan realesrgan rasterio scikit-image tqdm opencv-python

In [ ]:
# ── Cell 3: Clone GitHub Repository ──────────────────────────────────────
import os, sys
if not os.path.exists('/content/Depth-Wizard'):
    !git clone https://github.com/AjayBora002/Depth-Wizard.git /content/Depth-Wizard
%cd /content/Depth-Wizard/srm-project
sys.path.insert(0, '/content/Depth-Wizard/srm-project')
print('Repository ready at /content/Depth-Wizard/srm-project')

In [ ]:
# ── Cell 4: Generate Dense Training Pairs from Real Sentinel-2 Data ───────
from src.pair_generation import generate_all_pairs

print('Generating high-density synthetic pairs from Sentinel-2 tiles with overlap...')
n_pairs = generate_all_pairs(
    raw_dir='data/raw',
    out_dir='data/synthetic_pairs',
    patch_size=512,
    overlap=128,
    scale=4,
    rgb_only=False,
)
print(f'Done! Generated {n_pairs} multi-band training pairs.')

In [ ]:
# ── Cell 5: Run Deep Multi-Loss Training ──────────────────────────────────
from src.train import train

# Deep Training Configuration
results = train(
    pairs_dir='data/synthetic_pairs',
    output_dir='src/checkpoints',
    epochs=50,             # Set to 50 or 100 for deep convergence
    batch_size=8,          # Fast batching on T4 GPU
    crop_size=128,         # 128 LR -> 512 HR
    lr=1e-4,
    lambda_perceptual=0.10, # Semantic texture realism
    lambda_sam=0.05,        # Differentiable Spectral Angle Mapper
    lambda_edge=0.05,       # Sobel spatial gradient edge loss
    lambda_freq=0.02,       # 2D FFT Frequency magnitude loss
    save_every=10,
    num_workers=2,
    use_amp=True,           # Automatic Mixed Precision for high speed
)
print('Best checkpoint path:', results['best_checkpoint'])

In [ ]:
# ── Cell 6: Plot Training Loss & Validation PSNR Progress ─────────────────
import json
import matplotlib.pyplot as plt

with open('src/checkpoints/training_history.json') as f:
    hist = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(hist['train_loss'], label='Train Loss', color='#38bdf8')
ax1.plot(hist['val_loss'], label='Val Loss', color='#fb923c', linestyle='--')
ax1.set_title('Deep Multi-Loss Convergence')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

if 'val_psnr' in hist and hist['val_psnr']:
    ax2.plot(hist['val_psnr'], label='Val PSNR (dB)', color='#4ade80')
    ax2.set_title('Validation PSNR Progression')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('PSNR (dB)'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Download Checkpoint to Local Project ──────────────────────────
from google.colab import files
print('Downloading model_finetuned_best.pth and training_history.json...')
files.download('src/checkpoints/model_finetuned_best.pth')
files.download('src/checkpoints/training_history.json')
print('Drop the downloaded files into your local srm-project/src/checkpoints/ directory!')